# 02 - MTBF / MTTR Analysis
Reliability metrics and Weibull survival analysis.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Load Processed Metrics

In [ ]:
metrics = pd.read_csv("../data/processed/mtbf_metrics.csv")
metrics[["machine_id","vsm","mtbf_hrs","mttr_hrs","health_score","health_status"]].head(12)

## Health Distribution

In [ ]:
print(metrics["health_status"].value_counts().to_string())

## MTBF Distribution by VSM

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
colors = {"Alpha": "#2ECC71", "Beta": "#F39C12", "Gamma": "#E74C3C"}
for ax, vsm in zip(axes, ["Alpha", "Beta", "Gamma"]):
    data = metrics[metrics["vsm"] == vsm]["mtbf_hrs"]
    ax.hist(data, bins=10, color=colors[vsm], edgecolor="white")
    ax.set_title(f"VSM {vsm} MTBF")
    ax.set_xlabel("MTBF (hrs)")
axes[0].set_ylabel("Count")
plt.suptitle("MTBF Distributions by VSM Line", fontweight="bold")
plt.tight_layout()

## Health Score vs. MTBF

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for status, grp in metrics.groupby("health_status"):
    color = {"Healthy": "#2ECC71", "Monitor": "#F39C12", "Critical": "#E74C3C"}[status]
    ax.scatter(grp["mtbf_hrs"], grp["health_score"],
               c=color, label=status, alpha=0.85, s=90, edgecolors="white")
ax.set_xlabel("MTBF (hrs)")
ax.set_ylabel("Health Score (0-100)")
ax.legend()
plt.title("Health Score vs MTBF - Full Fleet")
plt.tight_layout()

## Weibull Analysis (CNC-A1)

- Shape β < 1 → infant mortality
- β ≈ 1 → random/exponential failures
- β > 1 → wear-out failures

In [ ]:
failures = pd.read_csv("../data/raw/failures.csv", parse_dates=["failure_date"])
cnc = failures[failures["machine_id"] == "CNC-A1"].sort_values("failure_date").copy()
cnc["iat_days"] = cnc["failure_date"].diff().dt.days
iat = cnc["iat_days"].dropna()
if len(iat) >= 4:
    shape, loc, scale = stats.weibull_min.fit(iat, floc=0)
    print(f"CNC-A1  |  beta (shape)={shape:.3f}  eta (scale)={scale:.1f} days")
    print(f"Failure regime: {'infant' if shape<0.9 else 'random' if shape<1.1 else 'wear-out'}")
else:
    print("Insufficient inter-arrival data for Weibull fit")

## Monthly Downtime Cost by VSM

In [ ]:
cost = metrics.groupby("vsm")["total_monthly_downtime_cost"].sum()
print(cost.apply(lambda x: f'${x:,.0f}').to_string())